# Arsitrad — Gemma 4 Inference Demo
## End-to-End: RAG + Fine-tuned Gemma 4 2B → Gradio UI

**Runtime:** Colab with **T4 GPU** (free tier sufficient)

---

## Contents
1. Setup & Install
2. Download LoRA Adapters + ChromaDB
3. RAG Retrieval
4. Model Inference
5. Gradio UI

**Author:** Hanif — UNDIP Architecture / Harizco Swarm


## 1. Setup & Installation


In [ ]:
#@title 1.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')


In [ ]:
#@title 1.2 — Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    gradio>=4.0.0 \
    huggingface_hub \
    tqdm
!pip install -q unsloth

# Upgrade chromadb to avoid version mismatch with persisted databases
!pip install -q --upgrade chromadb
print('All dependencies installed!')


In [ ]:
#@title 1.3 — Imports & Config
import os
import torch
from pathlib import Path

BASE_DIR   = Path('/content/arsitrad')
LORA_DIR   = BASE_DIR / 'lora_adapters'
CHROMA_DIR = BASE_DIR / 'chroma_db'

RAG_CONFIG = {
    'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'collection_name': 'arsitrad_national_regulations',
    'top_k': 5,
    'min_similarity': 0.3,
}

MODEL_CONFIG = {
    'max_seq_length': 2048,
    'load_in_4bit': True,
}
print('Config ready.')


## 2. Download LoRA Adapters + ChromaDB


In [ ]:
#@title 2.1 — Create directories
!mkdir -p /content/arsitrad/lora_adapters
!mkdir -p /content/arsitrad/chroma_db
print('Directories created.')


In [ ]:
#@title 2.2 — Download LoRA adapters from GitHub Release
import os, subprocess

LORA_URL = 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/arsitrad_finetuned_adapters.zip'
out = '/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters.zip'

if os.path.exists('/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters'):
    print('LoRA adapters already exist, skipping download.')
else:
    print('Downloading LoRA adapters (~747 MB)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', out, LORA_URL], check=True)
    print('Unzipping...')
    subprocess.run(['unzip', '-q', '-o', out, '-d', '/content/arsitrad/lora_adapters/'], check=True)
    print('Done!')


In [ ]:
#@title 2.3 — Download ChromaDB (national regulations — 22,649 chunks)
# ~262 MB zip — faster than individual files
import os, subprocess

CHROMA_ZIP = '/content/arsitrad/chroma_db_release.zip'
CHROMA_DIR = '/content/arsitrad/chroma_db'

if os.path.exists(CHROMA_DIR):
    print('ChromaDB already extracted, skipping.')
else:
    print('Downloading ChromaDB (~262 MB)...')
    subprocess.run(['wget', '-q', '--show-progress',
                    '-O', CHROMA_ZIP,
                    'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/chroma_db_release.zip'],
                   check=True)
    print('Extracting...')
    subprocess.run(['unzip', '-q', '-o', CHROMA_ZIP, '-d', '/content/arsitrad/'], check=True)
    print('Done!')


## 3. RAG Retrieval Pipeline


In [ ]:
#@title 3.1 — Initialize ChromaDB retriever
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

class RegulationRetriever:
    def __init__(self, persist_dir, collection_name):
        self.emb_model  = SentenceTransformer(RAG_CONFIG['embedding_model'])
        self.client     = chromadb.PersistentClient(path=persist_dir)
        self.collection = self.client.get_collection(name=collection_name)
        print(f'Collection "{collection_name}" — {self.collection.count()} chunks loaded.')

    def retrieve(self, query, top_k=5, min_similarity=0.3):
        raw_emb = self.emb_model.encode([query])[0]
        q_norm  = raw_emb / np.linalg.norm(raw_emb)
        results = self.collection.query(
            query_embeddings=[q_norm.tolist()],
            n_results=top_k,
            include=['documents','metadatas','embeddings'],
        )
        outputs = []
        for doc, metadata, emb in zip(results['documents'][0],
                                      results['metadatas'][0],
                                      results['embeddings'][0]):
            if emb is None:
                continue
            cos_sim = float(np.dot(q_norm, np.array(emb)) /
                          (np.linalg.norm(np.array(emb)) + 1e-8))
            if cos_sim >= min_similarity:
                outputs.append({'text': doc,
                                 'source': metadata.get('source','unknown'),
                                 'chunk_id': metadata.get('chunk_id',0),
                                 'similarity': round(cos_sim,4)})
        outputs.sort(key=lambda x: x['similarity'], reverse=True)
        return outputs

    def format_context(self, retrieved):
        if not retrieved:
            return 'No relevant regulations found.'
        parts = []
        for i, item in enumerate(retrieved, 1):
            parts.append(f"[{i}] {item['text']}\n(Source: {item['source']}, chunk {item['chunk_id']})")
        return '\n\n'.join(parts)

retriever = RegulationRetriever(str(CHROMA_DIR), RAG_CONFIG['collection_name'])


In [ ]:
#@title 3.2 — Test RAG retrieval
test_query = 'syarat teknis ventilasi alami pada rumah tinggal'
results   = retriever.retrieve(test_query, top_k=3)

print(f'Query: {test_query}\n')
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['source']} — {r['text'][:120]}...")
print(f'\nFormatted context:\n{retriever.format_context(results)[:400]}')


## 4. Load Fine-tuned Gemma 4 + Inference


In [ ]:
#@title 4.1 — Load LoRA adapters with Unsloth
# CRITICAL: If you ran Sections 2-3 first, do Runtime → Restart runtime
# before this cell. Clean GPU state prevents OOM.

import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

lora_path = str(LORA_DIR)

model, _ = FastLanguageModel.from_pretrained(
    model_name     = lora_path,
    max_seq_length = MODEL_CONFIG['max_seq_length'],
    load_in_4bit   = MODEL_CONFIG['load_in_4bit'],
    dtype          = None,
)
FastLanguageModel.for_inference(model)

tokenizer = AutoTokenizer.from_pretrained(lora_path)
print('Model + tokenizer loaded!')


In [ ]:
#@title 4.2 — Build prompt with RAG context
SYSTEM_PROMPT = (
    'Kamu adalah Arsitrad, asisten AI untuk regulasi dan saran arsitektur Indonesia. '
    'Jawablah akurat berdasarkan konteks peraturan yang diberikan. '
    'Jika informasi tidak cukup, katakan secara jujur.'
)

def build_prompt(user_query, context):
    if context and context != 'No relevant regulations found.':
        full_query = f'Konteks peraturan:\n{context}\n\nPertanyaan: {user_query}'
    else:
        full_query = user_query
    return (
        f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>user\n{full_query}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )


In [ ]:
#@title 4.3 — Run inference
def generate_response(user_query, max_new_tokens=512, temperature=0.7):
    retrieved = retriever.retrieve(user_query, top_k=RAG_CONFIG['top_k'])
    context   = retriever.format_context(retrieved)
    prompt    = build_prompt(user_query, context)
    inputs    = tokenizer(prompt, return_tensors='pt', truncation=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if '<start_of_turn>model\n' in response:
        response = response.split('<start_of_turn>model\n')[-1]
    if '<end_of_turn>' in response:
        response = response.split('<end_of_turn>')[0]
    return response.strip(), context, retrieved


In [ ]:
#@title 4.4 — Test inference with sample queries
test_queries = [
    'Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?',
    'Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?',
    'Strategi passive cooling untuk rumah di daerah tropis basah?',
]

for q in test_queries:
    print(f'\n{"="*60}\nQ: {q}\n{"-"*60}')
    response, context, retrieved = generate_response(q)
    print(f'R: {response[:600]}\n\n[Cited {len(retrieved)} regulation chunks]')


## 5. Gradio UI — Live Demo


In [ ]:
#@title 5.1 — Build & Launch Gradio Interface
import gradio as gr

def arsitrad_chat(message, history):
    response, context, retrieved = generate_response(message)
    sources = '\n'.join(f"- [{r['similarity']:.3f}] {r['source']}" for r in retrieved)
    return f"{response}\n\n---\n\n📚 Sources:\n{sources}"

demo = gr.ChatInterface(
    fn=arsitrad_chat,
    title='Arsitrad — AI Konsultan Hukum Bangunan Indonesia',
    description='Gemma 4 fine-tuned + RAG on 22,649 Indonesian construction law chunks.\n'
                'Modules: Disaster Damage | Building Permits | Settlement Upgrading | Passive Cooling',
    theme='default',
    multimodal_modal=False,
)

# share=True generates a public Colab URL for your thesis demo
demo.launch(server_name='0.0.0.0', server_port=7860, share=True)


---
## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Runtime → Restart runtime before inference (clears GPU memory from downloads) |
| Model not loading | Use LoRA path directly, or `google/gemma-4-E2B-it` for base model |
| ChromaDB empty | Re-run Section 2.3 — wget may timeout |
| Slow retrieval | Reduce `top_k` in RAG_CONFIG (e.g. 3 instead of 5) |
| Gradio link not working | Wait 30s after launch for Colab to provision the URL |
